In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from arch import arch_model
import plotly.graph_objects as go

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [2]:
df = pd.read_csv('dataset\\nasdq.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

prices = df['Close'].values
returns = pd.Series(np.diff(np.log(prices)) * 100)
dates = df['Date'].values[1:]

print(f'NASDAQ: {len(returns)} доходностей')
print(f'Период: {dates[0]} — {dates[-1]}')

NASDAQ: 3913 доходностей
Период: 2010-01-05T00:00:00.000000000 — 2024-10-25T00:00:00.000000000


In [3]:
# Ground truth variance (как в статье GINN)
rolling_mean = returns.rolling(window=90).mean()
raw_var = (returns - rolling_mean) ** 2
ground_truth_var = raw_var.rolling(window=5).mean()

gt_var = ground_truth_var.dropna().values
gt_var_log = np.log1p(gt_var)
gt_dates = dates[ground_truth_var.dropna().index]

print(f'GT variance: {len(gt_var)} точек')

GT variance: 3820 точек


In [4]:
# Окна
WINDOW = 90

X_windows, y_targets, window_dates = [], [], []
for i in range(WINDOW, len(gt_var_log)):
    X_windows.append(gt_var_log[i - WINDOW : i])
    y_targets.append(gt_var_log[i])
    window_dates.append(gt_dates[i])

X_windows = np.array(X_windows)
y_targets = np.array(y_targets)
window_dates = np.array(window_dates)

print(f'Окна: {X_windows.shape}')

Окна: (3730, 90)


In [5]:
# Split: train до 2018, val 2018-2020, test 2020+
window_dates_dt = window_dates.astype('datetime64')

train_idx = window_dates_dt < np.datetime64('2018-01-01')
val_idx = (window_dates_dt >= np.datetime64('2018-01-01')) & (window_dates_dt < np.datetime64('2020-01-01'))
test_idx = window_dates_dt >= np.datetime64('2020-01-01')

X_train, y_train = X_windows[train_idx], y_targets[train_idx]
X_val, y_val = X_windows[val_idx], y_targets[val_idx]
X_test, y_test = X_windows[test_idx], y_targets[test_idx]
dates_test = window_dates[test_idx]

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

Train: 1928, Val: 529, Test: 1273


In [6]:
# Нормализация
train_mean = X_train.mean()
train_std = X_train.std()

X_train_norm = (X_train - train_mean) / train_std
X_val_norm = (X_val - train_mean) / train_std
X_test_norm = (X_test - train_mean) / train_std
y_train_norm = (y_train - train_mean) / train_std
y_val_norm = (y_val - train_mean) / train_std
y_test_norm = (y_test - train_mean) / train_std

targets_denorm = np.expm1(y_test * train_std + train_mean)

print(f'Train mean: {train_mean:.4f}, std: {train_std:.4f}')

Train mean: 0.9160, std: 0.6094


In [7]:
print('Считаю GARCH для всех окон...')

garch_all_preds = []
for i in range(len(X_windows)):
    pos = ground_truth_var.dropna().index[WINDOW + i]
    train_data = returns.iloc[pos - 90 : pos]
    try:
        m = arch_model(train_data, vol='Garch', p=1, q=1, dist='Normal')
        r = m.fit(disp='off')
        f = r.forecast(horizon=1)
        garch_all_preds.append(f.variance.values[-1, 0])
    except:
        garch_all_preds.append(np.nan)
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(X_windows)}')

garch_all_preds = np.array(garch_all_preds)
nan_mask = np.isnan(garch_all_preds)
if nan_mask.sum() > 0:
    garch_all_preds[nan_mask] = np.nanmean(garch_all_preds)
    print(f'  Заполнено {nan_mask.sum()} NaN')

garch_log = np.log1p(garch_all_preds)
garch_all_norm = (garch_log - train_mean) / train_std

garch_train_norm = garch_all_norm[train_idx]
garch_val_norm = garch_all_norm[val_idx]
garch_test_norm = garch_all_norm[test_idx]
garch_test_raw = garch_all_preds[test_idx]

print(f'GARCH готов! {len(garch_all_preds)} предсказаний')

Считаю GARCH для всех окон...
  500/3730
  1000/3730
  1500/3730
  2000/3730
  2500/3730
  3000/3730
  3500/3730
GARCH готов! 3730 предсказаний


In [8]:
# LSTM (архитектура)
class VolatilityLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=256, num_layers=3, dropout=0.2):
        super(VolatilityLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.batch_norm = nn.BatchNorm1d(hidden_size)
        self.fc1 = nn.Linear(hidden_size, 128)
        self.fc2 = nn.Linear(128, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        normalized = self.batch_norm(last_output)
        fc1_out = self.relu(self.fc1(normalized))
        fc1_out = self.dropout(fc1_out)
        output = self.fc2(fc1_out)
        return output.squeeze(-1)


# Adaptive GINN (архитектура)
class FeatureGatingGINN(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.data_lstm = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, dropout=dropout)
        self.data_bn = nn.BatchNorm1d(hidden_size)
        self.garch_proj = nn.Sequential(
            nn.Linear(1, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, hidden_size)
        )
        self.lambda_lstm = nn.LSTM(input_size, 32, 2, batch_first=True, bidirectional=True)
        self.lambda_bn = nn.BatchNorm1d(64)
        self.lambda_head = nn.Sequential(
            nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x, garch_signal):
        out, _ = self.data_lstm(x)
        data_feat = self.data_bn(out[:, -1, :])
        garch_feat = self.garch_proj(garch_signal)
        l_out, _ = self.lambda_lstm(x)
        l_feat = self.lambda_bn(l_out[:, -1, :])
        lambda_val = self.lambda_head(l_feat)
        combined = data_feat + lambda_val * garch_feat
        output = self.output_head(combined).squeeze(-1)
        return output, lambda_val.squeeze(-1)

## 4. Datasets и DataLoaders

In [9]:
class VolatilityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class GatingDataset(Dataset):
    def __init__(self, X, y_real, y_garch):
        self.X = torch.FloatTensor(X).unsqueeze(-1)
        self.y_real = torch.FloatTensor(y_real)
        self.y_garch = torch.FloatTensor(y_garch)
    def __len__(self): return len(self.y_real)
    def __getitem__(self, idx): return self.X[idx], self.y_real[idx], self.y_garch[idx]

BS = 128

# LSTM loaders
lstm_train_ld = DataLoader(VolatilityDataset(X_train_norm, y_train_norm), BS, shuffle=True)
lstm_val_ld = DataLoader(VolatilityDataset(X_val_norm, y_val_norm), BS)
lstm_test_ld = DataLoader(VolatilityDataset(X_test_norm, y_test_norm), BS)

# GINN loaders
ginn_train_ld = DataLoader(GatingDataset(X_train_norm, y_train_norm, garch_train_norm), 256, shuffle=True)
ginn_val_ld = DataLoader(GatingDataset(X_val_norm, y_val_norm, garch_val_norm), BS)
ginn_test_ld = DataLoader(GatingDataset(X_test_norm, y_test_norm, garch_test_norm), BS)

print('DataLoaders готовы')

DataLoaders готовы


## 5. Обучение LSTM

In [22]:
lstm_model = VolatilityLSTM(hidden_size=256).to(device)
lstm_optimizer = optim.AdamW(lstm_model.parameters(), lr=1e-3)
lstm_scheduler = optim.lr_scheduler.ReduceLROnPlateau(lstm_optimizer, patience=20, factor=0.5)
criterion = nn.MSELoss()

best_val_loss = float('inf')
best_model_state = None
patience_counter = 0
PATIENCE = 30

for epoch in range(300):
    lstm_model.train()
    train_losses = []
    for X_batch, y_batch in lstm_train_ld:
        lstm_optimizer.zero_grad()
        preds = lstm_model(X_batch.to(device))
        loss = criterion(preds, y_batch.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        lstm_optimizer.step()
        train_losses.append(loss.item())

    lstm_model.eval()
    val_losses = []
    with torch.no_grad():
        for X_batch, y_batch in lstm_val_ld:
            preds = lstm_model(X_batch.to(device))
            val_losses.append(criterion(preds, y_batch.to(device)).item())

    avg_val = np.mean(val_losses)
    lstm_scheduler.step(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_model_state = {k: v.clone() for k, v in lstm_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 20 == 0:
        print(f'Эпоха {epoch+1}/300 - Train: {np.mean(train_losses):.6f}, Val: {avg_val:.6f}')

    if patience_counter >= PATIENCE:
        print(f'Ранняя остановка на эпохе {epoch+1}')
        break

lstm_model.load_state_dict(best_model_state)
print('LSTM обучен!')

Эпоха 20/300 - Train: 0.217426, Val: 0.128760
Эпоха 40/300 - Train: 0.197869, Val: 0.107682
Эпоха 60/300 - Train: 0.200300, Val: 0.117196
Эпоха 80/300 - Train: 0.163306, Val: 0.122128
Эпоха 100/300 - Train: 0.154588, Val: 0.093021
Эпоха 120/300 - Train: 0.163578, Val: 0.099738
Эпоха 140/300 - Train: 0.137967, Val: 0.092197
Эпоха 160/300 - Train: 0.148358, Val: 0.091170
Ранняя остановка на эпохе 168
LSTM обучен!


## 6. Обучение Adaptive GINN

In [23]:
def train_gating_model(model, train_loader, val_loader, epochs=300, patience=50):
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
    mse = nn.MSELoss()

    best_val = float('inf')
    best_state = None
    patience_cnt = 0

    for epoch in range(epochs):
        model.train()
        tr_loss, tr_lam = [], []
        for X, y_r, y_g in train_loader:
            X, y_r, y_g = X.to(device), y_r.to(device), y_g.to(device)
            optimizer.zero_grad()
            pred, lam = model(X, y_g.unsqueeze(-1))
            reg = 0.001 * ((lam - 0.5)**2).mean()
            loss = mse(pred, y_r) + reg
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss.append(loss.item())
            tr_lam.append(lam.mean().item())

        model.eval()
        val_loss, val_lam = [], []
        with torch.no_grad():
            for X, y_r, y_g in val_loader:
                X, y_r, y_g = X.to(device), y_r.to(device), y_g.to(device)
                pred, lam = model(X, y_g.unsqueeze(-1))
                val_loss.append(mse(pred, y_r).item())
                val_lam.append(lam.mean().item())

        avg_val = np.mean(val_loss)
        scheduler.step(avg_val)

        if avg_val < best_val:
            best_val = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1

        if (epoch+1) % 20 == 0:
            print(f'Ep {epoch+1:3d} | TrLoss={np.mean(tr_loss):.4f} | ValLoss={avg_val:.4f} | λ_tr={np.mean(tr_lam):.3f} | λ_val={np.mean(val_lam):.3f}')

        if patience_cnt >= patience:
            print(f'Early stop at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    return model


ginn_model = FeatureGatingGINN(hidden_size=256).to(device)
print(f'Model params: {sum(p.numel() for p in ginn_model.parameters()):,}')
ginn_model = train_gating_model(ginn_model, ginn_train_ld, ginn_val_ld, epochs=300)

Model params: 1,419,938
Ep  20 | TrLoss=0.2168 | ValLoss=0.1273 | λ_tr=0.563 | λ_val=0.547
Ep  40 | TrLoss=0.1815 | ValLoss=0.1286 | λ_tr=0.584 | λ_val=0.520
Ep  60 | TrLoss=0.1851 | ValLoss=0.1049 | λ_tr=0.582 | λ_val=0.555
Ep  80 | TrLoss=0.1767 | ValLoss=0.0954 | λ_tr=0.606 | λ_val=0.565
Ep 100 | TrLoss=0.1512 | ValLoss=0.1038 | λ_tr=0.596 | λ_val=0.537
Ep 120 | TrLoss=0.1463 | ValLoss=0.0975 | λ_tr=0.583 | λ_val=0.534
Ep 140 | TrLoss=0.1379 | ValLoss=0.0925 | λ_tr=0.578 | λ_val=0.492
Ep 160 | TrLoss=0.1330 | ValLoss=0.0921 | λ_tr=0.612 | λ_val=0.537
Ep 180 | TrLoss=0.1318 | ValLoss=0.0978 | λ_tr=0.629 | λ_val=0.562
Ep 200 | TrLoss=0.1277 | ValLoss=0.0905 | λ_tr=0.644 | λ_val=0.582
Early stop at epoch 211


## 7. Результаты

In [ ]:
# === Правильный таргет ===
targets_real = gt_var[-len(y_test):]

# === GARCH метрики ===
mask = ~np.isnan(garch_test_raw)
garch_mse = mean_squared_error(targets_real[mask], garch_test_raw[mask])
garch_mae = mean_absolute_error(targets_real[mask], garch_test_raw[mask])
garch_r2 = r2_score(targets_real[mask], garch_test_raw[mask])

# === LSTM метрики ===
lstm_model.eval()
lstm_preds = []
with torch.no_grad():
    for X_b, y_b in lstm_test_ld:
        lstm_preds.extend(lstm_model(X_b.to(device)).cpu().numpy())
lstm_preds = np.expm1(np.array(lstm_preds) * train_std + train_mean)
lstm_mse = mean_squared_error(targets_real, lstm_preds)
lstm_mae = mean_absolute_error(targets_real, lstm_preds)
lstm_r2 = r2_score(targets_real, lstm_preds)

# === Adaptive GINN метрики ===
ginn_model.eval()
ginn_preds, ginn_lambdas = [], []
with torch.no_grad():
    for X_b, y_r, y_g in ginn_test_ld:
        X_b, y_g = X_b.to(device), y_g.to(device)
        pred, lam = ginn_model(X_b, y_g.unsqueeze(-1))
        ginn_preds.extend(pred.cpu().numpy())
        ginn_lambdas.extend(lam.cpu().numpy())
ginn_preds = np.expm1(np.array(ginn_preds) * train_std + train_mean)
ginn_lambdas = np.array(ginn_lambdas)
ginn_mse = mean_squared_error(targets_real, ginn_preds)
ginn_mae = mean_absolute_error(targets_real, ginn_preds)
ginn_r2 = r2_score(targets_real, ginn_preds)

# === Таблица ===
print('=' * 60)
print('NASDAQ РЕЗУЛЬТАТЫ')
print('=' * 60)
print(f'{"Модель":<20} {"MSE":>10} {"MAE":>10} {"R²":>10}')
print('-' * 50) 
print(f'{"GARCH":<20} {garch_mse:>10.2f} {garch_mae:>10.4f} {garch_r2:>10.4f}')
print(f'{"LSTM":<20} {lstm_mse:>10.2f} {lstm_mae:>10.4f} {lstm_r2:>10.4f}')
print(f'{"Adaptive GINN":<20} {ginn_mse:>10.2f} {ginn_mae:>10.4f} {ginn_r2:>10.4f}')
print('=' * 60)
print(f'\nСреднее λ: {ginn_lambdas.mean():.4f} ± {ginn_lambdas.std():.4f}')

NASDAQ РЕЗУЛЬТАТЫ
Модель                      MSE        MAE         R²
--------------------------------------------------
GARCH                     10.12     1.4775     0.8301
LSTM                       7.02     0.7461     0.8821
Adaptive GINN              8.51     0.7693     0.8572

Среднее λ: 0.6528 ± 0.1509


In [28]:
# График: все модели
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates_test, y=targets_denorm, name='Реальность', opacity=0.4))
fig.add_trace(go.Scatter(x=dates_test, y=garch_test_raw, name='GARCH', opacity=0.7))
fig.add_trace(go.Scatter(x=dates_test, y=lstm_preds, name='LSTM', opacity=0.7))
fig.add_trace(go.Scatter(x=dates_test, y=ginn_preds, name='Adaptive GINN', opacity=0.7))
fig.update_layout(title='NASDAQ: все модели vs реальность',
                  xaxis_title='Дата', yaxis_title='σ²')
fig.show()

In [29]:
# График: адаптивный λ
fig_lam = go.Figure()
fig_lam.add_trace(go.Scatter(x=dates_test, y=ginn_lambdas, name='λ(t)', line=dict(color='purple')))
fig_lam.add_hline(y=0.5, line_dash='dash', line_color='green', annotation_text='Balance λ=0.5')
fig_lam.update_layout(title='NASDAQ: Адаптивный λ(t) по времени',
                      xaxis_title='Дата', yaxis_title='λ')
fig_lam.show()